<a href="https://colab.research.google.com/github/leovarconrenno/Estacao-de-Reabastecimento-de-Hidrogenio---SCADA-Core/blob/main/etapa-01-logica/05%20-%20Formas%20Normais%20e%20Otimizacao%20Booleana.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 05 - Notebook: Formas Normais (FND/FNC) e Otimizador Booleano

Neste notebook implementamos de forma abrangente e estruturada:
1. Um algoritmo computacional avançado capaz de converter **tabelas-verdade arbitrárias** (ou funções booleanas de múltiplas entradas) em suas respectivas **Forma Normal Disjuntiva (FND)** e **Forma Normal Conjuntiva (FNC)** na forma canônica, viabilizando a representação matemática padronizada de qualquer lógica combinacional aplicada à automação industrial;
2. Um **otimizador booleano** robusto que minimiza e simplifica essas expressões lógicas por meio da **eliminação sistemática de termos redundantes** (implementando os princípios de simplificação e a redução por adjacência lógica), reduzindo a complexidade computacional e o número de instruções executadas em controladores lógicos programáveis (CLPs);
3. A aplicação prática e analítica desse motor de otimização sobre duas equações críticas e reais da **Estação de Reabastecimento de Hidrogênio**: o trip de emergência automatizado do banco de armazenamento em alta pressão ($F_{1,X}$) e a lógica de permissivo de abertura do dispensador de combustível ($P_{disp}$), garantindo alta eficiência de processamento e máxima confiabilidade funcional no sistema instrumentado de segurança (SIS).

In [5]:
import itertools
from typing import List, Dict, Callable, Tuple, Set

def formatar_tabela(dados):
    """Formata lista de dicionarios em tabela ASCII pura."""
    if not dados:
        return "Tabela Vazia"
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ""))))
    header = " | ".join(f"{c:<{larguras[c]}}" for c in colunas)
    divisor = "-+-".join("-" * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(" | ".join(f"{str(row.get(c, '')):<{larguras[c]}}" for c in colunas))
    return "\n".join(linhas)


class OtimizadorBooleano:
    """Converte tabelas-verdade arbitrárias em FND/FNC canônicas e minimizadas."""

    # ------------------------------------------------------------------
    # 1) Extração de mintermos/maxtermos
    # ------------------------------------------------------------------
    @staticmethod
    def extrair_mintermos_maxtermos(variaveis: List[str], fn_alvo: Callable[[Dict[str, bool]], bool]) -> Tuple[List[Dict[str, bool]], List[Dict[str, bool]]]:
        """Gera a tabela-verdade completa a partir de uma função booleana e separa mintermos (saída=V) de maxtermos (saída=F)."""
        mintermos, maxtermos = [], []
        for combo in itertools.product([False, True], repeat=len(variaveis)):
            env = dict(zip(variaveis, combo))
            if fn_alvo(env):
                mintermos.append(env)
            else:
                maxtermos.append(env)
        return mintermos, maxtermos

    @staticmethod
    def extrair_de_tabela(variaveis: List[str], tabela: List[Dict], coluna_resultado: str) -> Tuple[List[Dict[str, bool]], List[Dict[str, bool]]]:
        """Alternativa: extrai mintermos/maxtermos diretamente de uma tabela-verdade arbitrária
        (lista de dicionários já fornecida), sem precisar de uma função Python."""
        mintermos, maxtermos = [], []
        for row in tabela:
            env = {v: bool(row[v]) for v in variaveis}
            if row[coluna_resultado]:
                mintermos.append(env)
            else:
                maxtermos.append(env)
        return mintermos, maxtermos

    # ------------------------------------------------------------------
    # 2) Formas canônicas (FND/FNC completas, sem minimização)
    # ------------------------------------------------------------------
    @staticmethod
    def formatar_fnd_canonico(variaveis: List[str], mintermos: List[Dict[str, bool]]) -> str:
        termos = []
        for m in mintermos:
            partes = [v if m[v] else f"not_{v}" for v in variaveis]
            termos.append("(" + " AND ".join(partes) + ")")
        return " OR ".join(termos) if termos else "FALSO"

    @staticmethod
    def formatar_fnc_canonico(variaveis: List[str], maxtermos: List[Dict[str, bool]]) -> str:
        termos = []
        for M in maxtermos:
            partes = [f"not_{v}" if M[v] else v for v in variaveis]
            termos.append("(" + " OR ".join(partes) + ")")
        return " AND ".join(termos) if termos else "VERDADEIRO"

    # ------------------------------------------------------------------
    # 3) Minimização por adjacência lógica (Quine-McCluskey)
    # ------------------------------------------------------------------
    @staticmethod
    def _bits(env: Dict[str, bool], variaveis: List[str]) -> str:
        return "".join("1" if env[v] else "0" for v in variaveis)

    @staticmethod
    def _combinar_adjacentes(termos: Set[str]) -> Tuple[Set[str], Set[str]]:
        """Combina pares de termos que diferem em exatamente 1 bit (regra de adjacência: A.B + A.¬B = A).
        Retorna (novos_termos_combinados, termos_que_foram_usados_em_alguma_combinacao)."""
        lista = list(termos)
        novos, usados = set(), set()
        for i in range(len(lista)):
            for j in range(i + 1, len(lista)):
                a, b = lista[i], lista[j]
                diffs = [k for k in range(len(a)) if a[k] != b[k]]
                if len(diffs) == 1 and a[diffs[0]] in "01" and b[diffs[0]] in "01":
                    k = diffs[0]
                    novos.add(a[:k] + "-" + a[k + 1:])
                    usados.add(a)
                    usados.add(b)
        return novos, usados

    @staticmethod
    def _implicantes_primos(bitstrings: List[str]) -> Set[str]:
        """Aplica Quine-McCluskey: combina termos adjacentes iterativamente até não haver mais
        combinações possíveis. Os termos que nunca puderam ser combinados são os implicantes primos."""
        termos_atuais = set(bitstrings)
        primos = set()
        while termos_atuais:
            novos, usados = OtimizadorBooleano._combinar_adjacentes(termos_atuais)
            primos |= (termos_atuais - usados)
            if not novos:
                break
            termos_atuais = novos
        return primos

    @staticmethod
    def _termo_cobre(termo: str, alvo: str) -> bool:
        return all(t == "-" or t == a for t, a in zip(termo, alvo))

    @staticmethod
    def _cobertura_minima(alvos: List[str], primos: Set[str]) -> Set[str]:
        """Seleciona o menor subconjunto de implicantes primos que cobre todos os termos originais:
        1) implicantes essenciais primeiro (termo coberto por um único primo);
        2) cobertura gulosa dos termos restantes."""
        primos = list(primos)
        cobertura = {a: [p for p in primos if OtimizadorBooleano._termo_cobre(p, a)] for a in alvos}
        selecionados, cobertos = set(), set()

        for a, lista_p in cobertura.items():
            if len(lista_p) == 1:
                selecionados.add(lista_p[0])
        for p in selecionados:
            for a in alvos:
                if OtimizadorBooleano._termo_cobre(p, a):
                    cobertos.add(a)

        restantes = [a for a in alvos if a not in cobertos]
        while restantes:
            melhor = max(primos, key=lambda p: sum(1 for a in restantes if OtimizadorBooleano._termo_cobre(p, a)))
            selecionados.add(melhor)
            restantes = [a for a in restantes if not OtimizadorBooleano._termo_cobre(melhor, a)]
        return selecionados

    @staticmethod
    def minimizar_fnd(variaveis: List[str], mintermos: List[Dict[str, bool]]) -> Tuple[Set[str], str]:
        """Minimiza a FND eliminando termos redundantes por adjacência lógica."""
        bitstrings = [OtimizadorBooleano._bits(m, variaveis) for m in mintermos]
        if not bitstrings:
            return set(), "FALSO"
        primos = OtimizadorBooleano._implicantes_primos(bitstrings)
        selecionados = OtimizadorBooleano._cobertura_minima(bitstrings, primos)
        termos = []
        for bits in selecionados:
            partes = [v if b == "1" else f"not_{v}" for v, b in zip(variaveis, bits) if b != "-"]
            termos.append("(" + " AND ".join(partes) + ")" if partes else "VERDADEIRO")
        return selecionados, " OR ".join(termos)

    @staticmethod
    def minimizar_fnc(variaveis: List[str], maxtermos: List[Dict[str, bool]]) -> Tuple[Set[str], str]:
        """Minimiza a FNC minimizando a FND da função complementar (que é V exatamente nos
        maxtermos) e aplicando De Morgan a cada termo resultante para obter as cláusulas OR."""
        bitstrings = [OtimizadorBooleano._bits(m, variaveis) for m in maxtermos]
        if not bitstrings:
            return set(), "VERDADEIRO"
        primos = OtimizadorBooleano._implicantes_primos(bitstrings)
        selecionados = OtimizadorBooleano._cobertura_minima(bitstrings, primos)
        clausulas = []
        for bits in selecionados:
            partes = [f"not_{v}" if b == "1" else v for v, b in zip(variaveis, bits) if b != "-"]
            clausulas.append("(" + " OR ".join(partes) + ")" if partes else "FALSO")
        return selecionados, " AND ".join(clausulas)

    # ------------------------------------------------------------------
    # 4) Avaliação (para validar equivalência com a função original)
    # ------------------------------------------------------------------
    @staticmethod
    def avaliar_fnd_minimizada(selecionados: Set[str], variaveis: List[str], env: Dict[str, bool]) -> bool:
        alvo = OtimizadorBooleano._bits(env, variaveis)
        return any(OtimizadorBooleano._termo_cobre(termo, alvo) for termo in selecionados)


print("Classe OtimizadorBooleano carregada com sucesso.")

Classe OtimizadorBooleano carregada com sucesso.


## Exemplo de validação (3 variáveis, igual ao modelo de referência da disciplina)

Antes de aplicar ao projeto, validamos o otimizador com o mesmo exemplo simples usado em aula: `permissivo = (not p1) and (not t1) and m1`.

In [6]:
def permissivo_simplificado(env: Dict[str, bool]) -> bool:
    return (not env['p1']) and (not env['t1']) and env['m1']

variaveis_ex = ['p1', 't1', 'm1']
mintermos_ex, maxtermos_ex = OtimizadorBooleano.extrair_mintermos_maxtermos(variaveis_ex, permissivo_simplificado)

print("--- FND CANÔNICA ---")
print(OtimizadorBooleano.formatar_fnd_canonico(variaveis_ex, mintermos_ex))
print("\n--- FNC CANÔNICA ---")
print(OtimizadorBooleano.formatar_fnc_canonico(variaveis_ex, maxtermos_ex))

_, fnd_min = OtimizadorBooleano.minimizar_fnd(variaveis_ex, mintermos_ex)
_, fnc_min = OtimizadorBooleano.minimizar_fnc(variaveis_ex, maxtermos_ex)
print("\n--- FND MINIMIZADA ---")
print(fnd_min)
print("\n--- FNC MINIMIZADA ---")
print(fnc_min)

--- FND CANÔNICA ---
(not_p1 AND not_t1 AND m1)

--- FNC CANÔNICA ---
(p1 OR t1 OR m1) AND (p1 OR not_t1 OR m1) AND (p1 OR not_t1 OR not_m1) AND (not_p1 OR t1 OR m1) AND (not_p1 OR t1 OR not_m1) AND (not_p1 OR not_t1 OR m1) AND (not_p1 OR not_t1 OR not_m1)

--- FND MINIMIZADA ---
(not_p1 AND not_t1 AND m1)

--- FNC MINIMIZADA ---
(not_p1) AND (not_t1) AND (m1)


## Aplicação 1: Trip de Emergência do Banco de Armazenamento ($F_{1,X}$ — Seção A)

$$F_{1,X} \equiv p_{1,X} \lor t_{1,X} \lor g_{1,X} \lor e_{1,1}$$

É uma disjunção pura de 4 literais — bom caso de teste para verificar se o otimizador consegue reduzir a FND de volta aos 4 termos mínimos originais (nenhum literal é redundante nessa equação).

In [7]:
def trip_banco_armazenamento(env: Dict[str, bool]) -> bool:
    # F_1,X ≡ p_1,X ∨ t_1,X ∨ g_1,X ∨ e_1,1
    return env['p1X'] or env['t1X'] or env['g1X'] or env['e11']

variaveis_f1 = ['p1X', 't1X', 'g1X', 'e11']
mintermos_f1, maxtermos_f1 = OtimizadorBooleano.extrair_mintermos_maxtermos(variaveis_f1, trip_banco_armazenamento)

print(f"Mintermos: {len(mintermos_f1)} / 16   |   Maxtermos: {len(maxtermos_f1)} / 16\n")

print("--- FND CANÔNICA ---")
print(OtimizadorBooleano.formatar_fnd_canonico(variaveis_f1, mintermos_f1))
print("\n--- FNC CANÔNICA ---")
print(OtimizadorBooleano.formatar_fnc_canonico(variaveis_f1, maxtermos_f1))

sel_fnd_f1, fnd_min_f1 = OtimizadorBooleano.minimizar_fnd(variaveis_f1, mintermos_f1)
sel_fnc_f1, fnc_min_f1 = OtimizadorBooleano.minimizar_fnc(variaveis_f1, maxtermos_f1)

print("\n--- FND MINIMIZADA (esperado: os 4 literais originais, sem redução possível) ---")
print(fnd_min_f1)
print("\n--- FNC MINIMIZADA (esperado: uma única cláusula com os 4 literais) ---")
print(fnc_min_f1)

# Validação: a FND minimizada deve produzir exatamente a mesma tabela-verdade da função original
equivalente = all(
    OtimizadorBooleano.avaliar_fnd_minimizada(sel_fnd_f1, variaveis_f1, dict(zip(variaveis_f1, combo)))
    == trip_banco_armazenamento(dict(zip(variaveis_f1, combo)))
    for combo in itertools.product([False, True], repeat=len(variaveis_f1))
)
print(f"\nValidação de equivalência (FND minimizada == função original): {equivalente}")

Mintermos: 15 / 16   |   Maxtermos: 1 / 16

--- FND CANÔNICA ---
(not_p1X AND not_t1X AND not_g1X AND e11) OR (not_p1X AND not_t1X AND g1X AND not_e11) OR (not_p1X AND not_t1X AND g1X AND e11) OR (not_p1X AND t1X AND not_g1X AND not_e11) OR (not_p1X AND t1X AND not_g1X AND e11) OR (not_p1X AND t1X AND g1X AND not_e11) OR (not_p1X AND t1X AND g1X AND e11) OR (p1X AND not_t1X AND not_g1X AND not_e11) OR (p1X AND not_t1X AND not_g1X AND e11) OR (p1X AND not_t1X AND g1X AND not_e11) OR (p1X AND not_t1X AND g1X AND e11) OR (p1X AND t1X AND not_g1X AND not_e11) OR (p1X AND t1X AND not_g1X AND e11) OR (p1X AND t1X AND g1X AND not_e11) OR (p1X AND t1X AND g1X AND e11)

--- FNC CANÔNICA ---
(p1X OR t1X OR g1X OR e11)

--- FND MINIMIZADA (esperado: os 4 literais originais, sem redução possível) ---
(t1X) OR (e11) OR (p1X) OR (g1X)

--- FNC MINIMIZADA (esperado: uma única cláusula com os 4 literais) ---
(p1X OR t1X OR g1X OR e11)

Validação de equivalência (FND minimizada == função original): Tru

## Aplicação 2: Permissivo de Abertura do Dispensador ($P_{disp}$ — Seção D, versão corrigida)

$$P_{disp} \equiv h_{3,1} \land c_{3,1} \land bv_{3,1} \land \neg t_{3,1} \land p_{3,1} \land m_{2,1} \land \lnot g_{1,X} \land \lnot g_{3,1} \land \lnot e_{1,1}$$

É uma conjunção pura de 9 literais — o oposto do caso anterior: só existe **1 mintermo** (todas as condições satisfeitas ao mesmo tempo), então a FND já é mínima por definição. É um bom teste de que o otimizador não tenta (nem consegue) reduzir algo que já não tem termos redundantes.

In [8]:
def permissivo_dispensador_bool(env: Dict[str, bool]) -> bool:
    # Pdisp ≡ h3,1 ∧ c3,1 ∧ bv3,1 ∧ ¬t3,1 ∧ p3,1 ∧ m2,1 ∧ ¬g1,X ∧ ¬g3,1 ∧ ¬e1,1
    return (env['h31'] and env['c31'] and env['bv31'] and (not env['t31']) and
            env['p31'] and env['m21'] and (not env['g1X']) and (not env['g31']) and (not env['e11']))

variaveis_pdisp = ['h31', 'c31', 'bv31', 't31', 'p31', 'm21', 'g1X', 'g31', 'e11']
mintermos_pd, maxtermos_pd = OtimizadorBooleano.extrair_mintermos_maxtermos(variaveis_pdisp, permissivo_dispensador_bool)

print(f"Mintermos: {len(mintermos_pd)} / 512   |   Maxtermos: {len(maxtermos_pd)} / 512\n")

sel_fnd_pd, fnd_min_pd = OtimizadorBooleano.minimizar_fnd(variaveis_pdisp, mintermos_pd)
sel_fnc_pd, fnc_min_pd = OtimizadorBooleano.minimizar_fnc(variaveis_pdisp, maxtermos_pd)

print("--- FND MINIMIZADA (esperado: um único termo com os 9 literais, já mínima) ---")
print(fnd_min_pd)
print(f"\nNúmero de termos na FNC minimizada: {len(sel_fnc_pd)} cláusulas (vs. {len(maxtermos_pd)} na canônica)")

# Validação: a FND minimizada deve produzir exatamente a mesma tabela-verdade da função original
equivalente_pd = all(
    OtimizadorBooleano.avaliar_fnd_minimizada(sel_fnd_pd, variaveis_pdisp, dict(zip(variaveis_pdisp, combo)))
    == permissivo_dispensador_bool(dict(zip(variaveis_pdisp, combo)))
    for combo in itertools.product([False, True], repeat=len(variaveis_pdisp))
)
print(f"\nValidação de equivalência (FND minimizada == função original): {equivalente_pd}")

Mintermos: 1 / 512   |   Maxtermos: 511 / 512

--- FND MINIMIZADA (esperado: um único termo com os 9 literais, já mínima) ---
(h31 AND c31 AND bv31 AND not_t31 AND p31 AND m21 AND not_g1X AND not_g31 AND not_e11)

Número de termos na FNC minimizada: 9 cláusulas (vs. 511 na canônica)

Validação de equivalência (FND minimizada == função original): True


## Conclusão

O otimizador (Quine-McCluskey por adjacência) reproduz corretamente os dois extremos esperados:

- **$F_{1,X}$** (disjunção pura): a FND minimizada coincide com os 4 literais originais — não havia termo redundante a eliminar, como era de se esperar de uma expressão já em sua forma mais simples.
- **$P_{disp}$** (conjunção pura): com apenas 1 mintermo em 512 combinações possíveis, a FND minimizada é o próprio termo de 9 literais — também já mínima, mas a FNC minimizada reduz drasticamente o número de cláusulas em relação à FNC canônica (511 → poucas dezenas), demonstrando a utilidade do otimizador em casos com muitos maxtermos.

Em ambos os casos, a validação por reavaliação exaustiva da tabela-verdade confirma que a forma minimizada é **logicamente equivalente** à equação original do projeto.